<a href="https://colab.research.google.com/github/Lattemelia/Portfolio/blob/main/%E3%80%900624%EF%BD%9E%E7%8F%BE%E5%9C%A8%E4%BD%9C%E6%88%90%E4%B8%AD%E3%80%91/%E4%B8%AD%E5%8F%A4%E8%BB%8A%E3%81%AE%E4%BE%A1%E6%A0%BC%E4%BA%88%E6%B8%AC(KaggleConpe).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 新しいセクション

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
PATH = r"/content/drive/MyDrive/Colab Notebooks"

test_df = pd.read_csv(PATH + r"/test.csv")
train_df = pd.read_csv(PATH + r"/train.csv")

marged_df = pd.concat([test_df,train_df],axis=0)

In [28]:
marged_df.isnull().sum()

,0
id,0
brand,0
model,0
model_year,0
milage,0
fuel_type,8466
engine,0
transmission,0
ext_col,0
int_col,0


In [5]:
cols_to_lower = ['brand', 'model', 'engine', 'fuel_type', 'transmission']

for col in cols_to_lower:
    marged_df[col] = marged_df[col].str.lower().str.strip()

In [6]:
# (?i) を先頭に配置することで、パターン全体の大文字小文字を無視します
hp_pattern = r"(?i)(\d+\.\d+|\d+)\s*hp"

# engine列の馬力表記を丸ごと削除
marged_df["engine"] = (
    marged_df["engine"]
    .astype(str)
    .str.replace(hp_pattern, "", regex=True)
    .str.strip()
)

# 連続したスペースを1つに統合
marged_df["engine"] = marged_df["engine"].str.replace(r"\s+", " ", regex=True)

In [25]:
import pandas as pd


def extract_model_by_word_count(model_str):
    if pd.isna(model_str):
        return None

    # 文字列を前後の空白を消して、スペースで分割（単語のリストにする）
    words = str(model_str).strip().split()
    word_count = len(words)

    if word_count == 0:
        return None
    # 1単語のみの場合（例: "Passat"）は、そのままその単語を返す
    elif word_count == 1:
        return words[0]
    # 2単語の場合（例: "A3 2.0T" ➔ "A3 2.0T"）は、そのまま2単語結合して返す
    elif word_count == 1:
        return " ".join(words)
    # 3単語以上の場合（例: "Golf Alltrack TSI SE" ➔ "Golf Alltrack"）は、最初の2単語を返す
    else:
        return " ".join(words[:2])


# ─── marged_df への適用 ───
# model列にルールを適用し、新しく 'pure_model' カラムを作成します
marged_df["pure_model"] = marged_df["model"].apply(extract_model_by_word_count)

marged_df["pure_model"].value_counts()

,count
pure_model,
rover range,12795
911 carrera,6457
f-150 xlt,5353
e-class e,5204
corvette stingray,4104
...,...
forte lx,2
x5 xdrive,2
v60 t6,2


In [8]:
import re
import numpy as np
import pandas as pd

# ─── 1. 排気量を抽出するための正規表現パターン ───
disp_pattern = r"(?i)(\d+\.\d+|\d+)\s*(l|liter|liters|litres)"


# ─── 2. 抽出用の関数を定義 ───
def extract_displacement(engine_text):
    if pd.isna(engine_text):
        return np.nan

    match = re.search(disp_pattern, str(engine_text))
    if match:
        try:
            # マッチした最初のグループ（数値部分）をfloat型に変換して返す
            return float(match.group(1))
        except ValueError:
            return np.nan
    return np.nan


# ─── 3. marged_df への適用（抽出） ───
# engine列から排気量を抜き出し、新列 'displacement_l' を作成
marged_df["displacement_l"] = marged_df["engine"].apply(extract_displacement)

#排気量以外の情報を'engine_type'へ入力
marged_df["engine_type"] = (
    marged_df["engine"]
    .astype(str)
    .str.replace(disp_pattern, "", regex=True)
    .str.strip()
)

# 文字を消した後に発生する「2連スペース」を1つのスペースに美しく統合
marged_df["engine"] = marged_df["engine"].str.replace(r"\s+", " ", regex=True)

In [9]:
import numpy as np
import pandas as pd


def advanced_engine_classifier_v4(engine_str):
    # 最初から欠損しているものは None, NaN を返す
    if pd.isna(engine_str):
        return pd.Series([None, np.nan])

    s = str(engine_str).lower().strip()

    # ─── ① 【修正】電気自動車（EV）の判定 ───
    # 気筒数も排気量も「存在しない」ため、どちらも欠損値（None, np.nan）を返す
    if ("electric" in s or "battery" in s or "dual ac" in s) and not any(
        x in s for x in ["hybrid", "plug-in", "gas", "mild"]
    ):
        return pd.Series([None, np.nan])

    # ─── ② 気筒数キーワードによる判定 ───
    if any(x in s for x in ["12 cylinder", "v12", "w12", "w16"]):
        return pd.Series(["V12/W16", 6.0])

    if any(x in s for x in ["10 cylinder", "v10"]):
        return pd.Series(["V10", 5.2])

    if any(x in s for x in ["8 cylinder", "v8"]):
        return pd.Series(["V8", 4.0])

    if any(
        x in s
        for x in ["6 cylinder", "v6", "straight 6", "flat 6", "i6", "h6"]
    ):
        return pd.Series(["V6", 3.0])

    if any(x in s for x in ["5 cylinder", "i5"]):
        return pd.Series(["I5", 2.5])

    if (
        any(x in s for x in ["4 cylinder", "i4", "h4", "tsi", "tfsi", "gdi"])
        or "i-vtec v6" not in s
        and "i4" in s
    ):
        return pd.Series(["I4", 2.0])

    if any(x in s for x in ["3 cylinder", "i3"]):
        return pd.Series(["I3", 1.5])

    if "rotary" in s:
        return pd.Series(["Rotary", 1.3])

    if "hybrid" in s or "plug-in" in s:
        return pd.Series(["I4", 2.0])

    # ─── ③ 分からないものも None, np.nan を返す ───
    return pd.Series([None, np.nan])


# marged_dfへの一括適用
marged_df[["engine_type_clean", "displacement_l"]] = marged_df["engine"].apply(
    advanced_engine_classifier_v4
)

In [18]:
# 欠損値（NaN）も含めた種類数を確認したい場合
print("=== 📊 欠損値も含めた各カラムの種類数 ===")
for col in marged_df.columns:
    unique_count = len(marged_df[col].value_counts(dropna=False))
    print(f"{col:<20} : {unique_count} 種類")

=== 📊 欠損値も含めた各カラムの種類数 ===
id                   : 314223 種類
brand                : 57 種類
model                : 1894 種類
model_year           : 36 種類
milage               : 8440 種類
fuel_type            : 8 種類
engine               : 343 種類
transmission         : 51 種類
ext_col              : 319 種類
int_col              : 156 種類
accident             : 3 種類
clean_title          : 2 種類
price                : 1570 種類
pure_model           : 517 種類
displacement_l       : 9 種類
engine_type          : 129 種類
engine_type_clean    : 9 種類


In [20]:
marged_df["model_since"] = 2026 - marged_df["model_year"]

In [26]:
import pandas as pd

# ─── 1. price が入っている（欠損していない）行を Train にする ───
train_df = marged_df[marged_df["price"].notnull()].copy()

# ─── 2. price が欠損している行を Test にする ───
test_df = marged_df[marged_df["price"].isnull()].copy()

# ─── 3. Testデータから予測対象である 'price' 列を落とす（お作法） ───
test_df = test_df.drop(columns=["price"])

# 📊 分割後のデータ件数を確認
print(f"📁 元の全体のデータ数: {len(marged_df)}")
print(f"🚗 Train（学習用）のデータ数: {len(train_df)}")
print(f"🔮 Test（予測用）のデータ数 : {len(test_df)}")

📁 元の全体のデータ数: 314223
🚗 Train（学習用）のデータ数: 188533
🔮 Test（予測用）のデータ数 : 125690


In [27]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import root_mean_squared_error  # scikit-learn 1.4以降
from sklearn.model_selection import KFold

# ─── 1. 使用する特徴量（列）の定義 ───
features = [
    "brand",
    "model_since",
    "milage",
    "fuel_type",
    "transmission",
    "ext_col",
    "int_col",
    "accident",
    "clean_title",
    "pure_model",
    "displacement_l",
    "engine_type_clean",
]

# ─── 2. 文字列データを「category型」に一括変換 ───
# LightGBMに文字列をそのまま学習させるための必須の処理です
for col in features:
    if marged_df[col].dtype == "object":
        marged_df[col] = marged_df[col].astype("category")

# ─── 3. priceを基準にTrainとTestに切り分け ───
train_df = marged_df[marged_df["price"].notnull()].copy()
test_df = marged_df[marged_df["price"].isnull()].copy()

X = train_df[features]
y = train_df["price"]
X_test = test_df[features]

# ─── 4. 交差検証（5-Fold KFold）の準備 ───
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# スコアや予測結果を格納する箱
oof_preds = np.zeros(len(X))  # Out-of-Fold (検証データでの予測値)
test_preds = np.zeros(len(X_test))  # Testデータの予測値の平均用
cv_scores = []

# ─── 5. 交差検証ループの実行 ───
print("🚀 LightGBM 5-Fold Cross Validation 開始...\n" + "─" * 45)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    # データの分割
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    # LightGBM専用のデータセット形式に変換
    train_dataset = lgb.Dataset(X_train, label=y_train)
    val_dataset = lgb.Dataset(X_val, label=y_val, reference=train_dataset)

    # ハイパーパラメータの設定（タスクに応じて調整してください）
    params = {
        "objective": "regression",  # 回帰タスク
        "metric": "rmse",  # 評価指標：RMSE
        "learning_rate": 0.05,  # 学習率
        "num_leaves": 31,  # 木の複雑さ
        "random_state": 42 + fold,
        "verbose": -1,  # ログ出力を少し静かにする
    }

    # モデルの訓練
    model = lgb.train(
        params,
        train_dataset,
        num_boost_round=2000,
        valid_sets=[train_dataset, val_dataset],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(100),  # 100回ごとにログを表示
        ],
    )

    # 検証データでの予測（OOF）
    val_preds = model.predict(X_val, num_iteration=model.best_iteration)
    oof_preds[val_idx] = val_preds

    # テストデータの予測（各Foldのモデルで予測して、後で平均を取る）
    test_preds += model.predict(X_test, num_iteration=model.best_iteration) / 5

    # このFoldのスコア（RMSE）を計算
    fold_score = root_mean_squared_error(y_val, val_preds)
    cv_scores.append(fold_score)
    print(f"🔹 Fold {fold+1} RMSE: {fold_score:,.2f}")
    print("─" * 45)

# ─── 6. 全体の検証結果の表示 ───
oof_score = root_mean_squared_error(y, oof_preds)
print(f"🏆 5-Fold 平均 RMSE   : {np.mean(cv_scores):,.2f}")
print(f"🏆 全体 OOF  RMSE    : {oof_score:,.2f}")

# test_dfに予測結果を格納（これでいつでも提出や確認が可能です！）
test_df["predicted_price"] = test_preds

🚀 LightGBM 5-Fold Cross Validation 開始...
─────────────────────────────────────────────
🔹 Fold 1 RMSE: 68,498.96
─────────────────────────────────────────────
🔹 Fold 2 RMSE: 69,126.26
─────────────────────────────────────────────
[100]	training's rmse: 66507.8	valid_1's rmse: 74303.9
🔹 Fold 3 RMSE: 74,195.22
─────────────────────────────────────────────
[100]	training's rmse: 66409.9	valid_1's rmse: 76667.5
🔹 Fold 4 RMSE: 76,651.37
─────────────────────────────────────────────
🔹 Fold 5 RMSE: 76,832.00
─────────────────────────────────────────────
🏆 5-Fold 平均 RMSE   : 73,060.76
🏆 全体 OOF  RMSE    : 73,149.21
